# GCP + GitHub Actions Setup Guide
**Dr. Dave Wanik and Rohit Akole** -- Dept. Operations and Information Management -- University of Connecticut

This guide walks through setting up GCP permissions, service accounts, and GitHub Actions integration
for deploying Cloud Functions (Gen2) with Cloud Scheduler.

Follow these steps **in order** to get your own project running from the course template repo:
**[https://github.com/drdave-teaching/myscrapers-labs](https://github.com/drdave-teaching/myscrapers-labs)**

> **Before starting:** open Cloud Shell in your GCP console (top-right terminal icon).
> Fill in the Variables cell below -- everything else is derived automatically.


# 1. Variables -- Edit These First

Fill in the values below, then run this cell. **All other cells reference these variables.**
You only need to change the values in the EDITABLE section at the top.


In [ ]:
# ==============================================================
# EDITABLE -- fill these in, then run this cell
# ==============================================================
PROJECT_ID="YOUR_PROJECT_ID"                   # e.g. "jdoe-scraper-v1"
REGION="us-central1"                           # or nearest region e.g. us-east1
BUCKET_NAME="YOUR_BUCKET_NAME"                 # must be globally unique
GITHUB_REPO="YOUR_GITHUB_USERNAME/YOUR_REPO"   # e.g. "jdoe/myscrapers-labs"

# Scraper config -- also set these as GitHub Repository Variables (see Section 9)
BASE_SITE="https://newhaven.craigslist.org"    # base URL of the site to scrape
SEARCH_PATH="/search/cto"                      # cto=cars+trucks, apa=apartments, etc.
USER_AGENT="Mozilla/5.0 (compatible; research-bot/1.0)"
FUNCTION_NAME="listing-scraper"                # your GCP Cloud Function name
FUNCTION_DIR="cloud_function/scraper_cars"     # path to the function source in the repo

# ==============================================================
# AUTO-DERIVED -- do not edit below this line
# ==============================================================
RUNTIME_SA_ID="cf-runtime"
DEPLOYER_SA_ID="cf-deployer"
SCHED_SA_ID="cf-scheduler"
WIF_POOL="gh-pool"
WIF_PROVIDER="gh-provider"

RUNTIME_SA="${RUNTIME_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
DEPLOYER_SA="${DEPLOYER_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHED_SA="${SCHED_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"

# Get project number (needed for WIF and compute SA references)
PROJECT_NUMBER=$(gcloud projects describe "${PROJECT_ID}" --format="value(projectNumber)")

# Workload Identity principal set (links GitHub repo to GCP)
PRINCIPAL_SET="principalSet://iam.googleapis.com/projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/attribute.repository/${GITHUB_REPO}"

# Cloud Scheduler managed service agent
SCHEDULER_AGENT="service-${PROJECT_NUMBER}@gcp-sa-cloudscheduler.iam.gserviceaccount.com"

echo "Variables set for project: ${PROJECT_ID} (${PROJECT_NUMBER})"

# 2. Enable Required APIs

Safe to re-run -- `gcloud` skips APIs that are already enabled.


In [ ]:
gcloud services enable \
  cloudfunctions.googleapis.com \
  run.googleapis.com \
  cloudscheduler.googleapis.com \
  artifactregistry.googleapis.com \
  iam.googleapis.com \
  iamcredentials.googleapis.com \
  cloudbuild.googleapis.com \
  eventarc.googleapis.com \
  storage.googleapis.com \
  serviceusage.googleapis.com \
  cloudresourcemanager.googleapis.com \
  pubsub.googleapis.com \
  logging.googleapis.com \
  compute.googleapis.com

echo "APIs enabled."

# 3. Create Service Accounts

Three service accounts:
- **cf-runtime** -- the identity Cloud Functions run as
- **cf-deployer** -- the identity GitHub Actions uses to deploy
- **cf-scheduler** -- the identity Cloud Scheduler uses to invoke functions

The `|| true` means the command succeeds even if the SA already exists.


In [ ]:
gcloud iam service-accounts create "${RUNTIME_SA_ID}"  --display-name="Cloud Functions Runtime" || true
gcloud iam service-accounts create "${DEPLOYER_SA_ID}" --display-name="GitHub Deployer" || true
gcloud iam service-accounts create "${SCHED_SA_ID}"    --display-name="Scheduler Invoker" || true

gcloud iam service-accounts list

# 4. Workload Identity Federation (WIF) Setup

This is how GitHub Actions authenticates to GCP **without storing a JSON key anywhere**.
GCP trusts GitHub's OIDC token -- no secrets needed.

Run all three cells in this section in order.


In [ ]:
# Step 4a: Create the WIF pool (once per project)
gcloud iam workload-identity-pools create "${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub Actions Pool" || true


In [ ]:
# Step 4b: Create the OIDC provider (links GitHub to this pool)
gcloud iam workload-identity-pools providers create-oidc "${WIF_PROVIDER}" \
  --workload-identity-pool="${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub OIDC Provider" \
  --issuer-uri="https://token.actions.githubusercontent.com" \
  --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository,attribute.ref=assertion.ref" \
  --attribute-condition="attribute.repository=='${GITHUB_REPO}' && attribute.ref=='refs/heads/main'" || true


In [ ]:
# Step 4c: Allow GitHub Actions (for this repo) to impersonate the deployer SA
gcloud iam service-accounts add-iam-policy-binding "${DEPLOYER_SA}" \
  --role="roles/iam.workloadIdentityUser" \
  --member="${PRINCIPAL_SET}"


# 5. Project-Level Roles for the Deployer SA

The deployer SA needs permission to deploy Cloud Functions, manage Artifact Registry images,
manage Schedulers, and enable APIs.


In [ ]:
for ROLE in \
  roles/cloudfunctions.developer \
  roles/run.admin \
  roles/cloudscheduler.admin \
  roles/artifactregistry.writer \
  roles/serviceusage.serviceUsageAdmin; do
    gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
      --member="serviceAccount:${DEPLOYER_SA}" \
      --role="${ROLE}"
done

echo "Project roles granted to ${DEPLOYER_SA}"

# 6. Service Account Relationships

These bindings let the deployer SA act as the runtime SA (needed to deploy functions),
and let the Scheduler agent mint OIDC tokens for the runtime SA (needed to invoke functions on a schedule).


In [ ]:
# Allow deployer to deploy functions that run as the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"

# Allow deployer to administer the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountAdmin"

# Allow deployer to act as the default compute SA (needed for some Cloud Run operations)
gcloud iam service-accounts add-iam-policy-binding \
  "projects/-/serviceAccounts/${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"

# Allow Cloud Scheduler's managed agent to mint tokens to invoke the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${SCHEDULER_AGENT}" \
  --role="roles/iam.serviceAccountTokenCreator"

# Allow Cloud Scheduler SA (sched_sa) to invoke functions as the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${SCHED_SA}" \
  --role="roles/iam.serviceAccountTokenCreator"

echo "SA relationships configured."

# 7. Create Bucket and Grant Runtime SA Access

The scraper saves raw listing text here. The ETL and train functions read from it.
Bucket names must be globally unique -- if yours is taken, add a suffix like `-v2`.


In [ ]:
# Create the bucket if it doesn't already exist
if ! gcloud storage buckets describe "gs://${BUCKET_NAME}" >/dev/null 2>&1; then
  gcloud storage buckets create "gs://${BUCKET_NAME}" --project="${PROJECT_ID}" --location="${REGION}"
  echo "Bucket gs://${BUCKET_NAME} created."
else
  echo "Bucket gs://${BUCKET_NAME} already exists, skipping."
fi

# Grant runtime SA full object access (read + write)
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
  --member="serviceAccount:${RUNTIME_SA}" \
  --role="roles/storage.objectAdmin"

# Grant deployer SA read access (needed for sync-data workflow to push results to GitHub)
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role='roles/storage.objectViewer'

echo "Bucket permissions set."

# 8. Vertex AI Permissions (for LLM Extractor)

The LLM extractor Cloud Function calls Vertex AI (Gemini) to extract structured fields
from raw listing text. The runtime SA needs permission to call the Vertex AI API.


In [ ]:
# Grant Vertex AI User role to the runtime SA
gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
  --member="serviceAccount:${RUNTIME_SA}" \
  --role="roles/aiplatform.user"

echo "Vertex AI User role granted to ${RUNTIME_SA}"

# Extend the Cloud Run timeout for the LLM function (LLM calls can be slow on large batches)
gcloud run services update extractor-llm-poc \
  --region="${REGION}" \
  --timeout="3600s"   # 60 minutes max


# 9. Verify Everything

Run these to confirm all the pieces are in place before deploying.


In [ ]:
# List service accounts
gcloud iam service-accounts list

# Confirm WIF pool and provider exist
gcloud iam workload-identity-pools list --location=global
gcloud iam workload-identity-pools providers list \
  --workload-identity-pool="${WIF_POOL}" --location=global

# Check IAM bindings for the deployer SA
gcloud projects get-iam-policy "${PROJECT_ID}" | grep "${DEPLOYER_SA}" -A3

# Check IAM bindings on the runtime SA
gcloud iam service-accounts get-iam-policy "${RUNTIME_SA}"

# 10. Add Variables to GitHub Repository

Go to your forked repo -> **Settings** -> **Secrets and variables** -> **Actions** -> **Variables tab**

Add each of these as a Repository Variable (not a secret -- these are not sensitive).

| Variable | Value |
|---|---|
| `WORKLOAD_IDENTITY_PROVIDER` | `projects/PROJECT_NUMBER/locations/global/workloadIdentityPools/gh-pool/providers/gh-provider` |
| `DEPLOYER_SA` | `cf-deployer@YOUR_PROJECT_ID.iam.gserviceaccount.com` |
| `RUNTIME_SA` | `cf-runtime@YOUR_PROJECT_ID.iam.gserviceaccount.com` |
| `PROJECT_ID` | your GCP project ID |
| `REGION` | e.g. `us-central1` |
| `BUCKET_NAME` | your GCS bucket name |
| `FUNCTION_NAME` | e.g. `listing-scraper` |
| `FUNCTION_DIR` | `cloud_function/scraper_cars` |
| `BASE_SITE` | e.g. `https://newhaven.craigslist.org` |
| `SEARCH_PATH` | e.g. `/search/cto` |
| `USER_AGENT` | e.g. `Mozilla/5.0 (compatible; research-bot/1.0)` |

Run the copy/paste helper below to print the exact values from your Variables cell:


In [ ]:
# Copy/paste helper -- run this AFTER the Variables cell (Section 1) to print exact values
echo "--- Paste these into GitHub Repository Variables ---"
echo "WORKLOAD_IDENTITY_PROVIDER=projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/providers/${WIF_PROVIDER}"
echo "DEPLOYER_SA=${DEPLOYER_SA}"
echo "RUNTIME_SA=${RUNTIME_SA}"
echo "PROJECT_ID=${PROJECT_ID}"
echo "REGION=${REGION}"
echo "BUCKET_NAME=${BUCKET_NAME}"
echo "FUNCTION_NAME=${FUNCTION_NAME}"
echo "FUNCTION_DIR=${FUNCTION_DIR}"
echo "BASE_SITE=${BASE_SITE}"
echo "SEARCH_PATH=${SEARCH_PATH}"
echo "USER_AGENT=${USER_AGENT}"


# 11. Deploy and Test

Push to `main` to trigger the GitHub Actions workflows. Start with one at a time:

1. `.github/workflows/deploy-extractor.yml` -- simplest, good first test
2. `.github/workflows/deploy-materialize-master.yml`
3. `.github/workflows/deploy-train-dt.yml`
4. `.github/workflows/deploy-extractor-llm.yml` -- needs Vertex AI (Section 8)

Then confirm in GCP Console, or run:


In [ ]:
gcloud functions list --regions="${REGION}"
gcloud scheduler jobs list --location="${REGION}"

# Expected scheduler jobs after all deploys (job name = FUNCTION_NAME + -hourly):
# extractor-per-listing-hourly
# materialize-master-hourly
# train-dt-hourly
# listing-scraper-hourly   <- matches your FUNCTION_NAME variable


# 12. Architecture Summary

```
GitHub (OIDC)  --->  cf-deployer SA  (Workload Identity User)
                          |
                          |-- deploys Cloud Functions Gen2 + Scheduler
                          |
                          v
                     cf-runtime SA  (ServiceAccountUser)
                          |
                          |-- executes function logic
                          |-- reads/writes GCS bucket
                          |-- calls Vertex AI (LLM extractor)
                          |-- trusted by Scheduler agent (TokenCreator)
```

The `sync-data` workflow runs on a schedule and copies results from GCS back into
the `results/` folder of your GitHub repo so you can see model predictions without
opening the GCP console.


# 13. Fork and Go -- Your Starting Point

**[https://github.com/drdave-teaching/myscrapers-labs](https://github.com/drdave-teaching/myscrapers-labs)**

This repo contains:

| Folder | What it does |
|---|---|
| `cloud_function/scraper_cars/` | Fetches listing pages and saves raw text to GCS |
| `cloud_function/extractor-per-listing/` | RegEx ETL -- extracts price, year, mileage etc. |
| `cloud_function/extractor-llm-poc/` | LLM extraction via Vertex AI (Gemini) |
| `cloud_function/materialize-master/` | Combines per-run CSVs into a master dataset |
| `cloud_function/train-dt/` | Trains a Decision Tree on the materialized data |
| `.github/workflows/` | CI/CD -- push to main, GCP deploys automatically |
| `results/` | Model predictions synced from GCS back to GitHub |

**For your project:** change `BASE_SITE` and `SEARCH_PATH` to scrape something other than cars.
The pipeline works the same for apartments, jobs, furniture -- anything on a listings site.
